In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "gold")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
from pyspark.sql.functions import current_timestamp, col

# List of CSV file names we are using
csv_files = ["customers", "order_items", "orders", "products"]

for file in csv_files:
    # 1. Read from the CSV file
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"/Volumes/{catalog}/bronze/raw_data/Datasets/{file}.csv")
    # 2. Add Ingestion Metadata (Standard Bronze Practice)
    df_bronze = df.withColumn("ingestion_date", current_timestamp()) \
                  .withColumn("source_path", col("_metadata.file_path"))
    
    # 3. Write to your Bronze Schema
    df_bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"my_assessment.bronze.{file}")
    
    print(f"CSV file {file} successfully ingested to Bronze.")